In [48]:
import json
import pandas as pd
import random
from pathlib import Path

# Set random seed for reproducibility
random.seed(42)

# Define paths
datasets_dir = Path(r"c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\datasets")
csc_path = datasets_dir / "CSC_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_filtered.json"
multipico_path = datasets_dir / "MultiPICo_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_filtered.json"
output_path = Path(r"c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\human") / "irony_triggering_detection_human_ann.xlsx"

print("Loading datasets...")

# Load CSC dataset
with open(csc_path, 'r', encoding='utf-8') as f:
    csc_data = json.load(f)

# Load MultiPICo dataset
with open(multipico_path, 'r', encoding='utf-8') as f:
    multipico_data = json.load(f)

print(f"CSC entries loaded: {len(csc_data['entries'])}")
print(f"MultiPICo entries loaded: {len(multipico_data['entries'])}")

Loading datasets...
CSC entries loaded: 2946
MultiPICo entries loaded: 1382
CSC entries loaded: 2946
MultiPICo entries loaded: 1382


In [49]:
# Process CSC dataset
print("\nProcessing CSC dataset...")

# Separate entries by aggregate_sarcasm_label (inside annotations)
csc_label_1 = [entry for entry in csc_data['entries'] if entry.get('annotations', {}).get('aggregate_sarcasm_label') == 1]
csc_label_0 = [entry for entry in csc_data['entries'] if entry.get('annotations', {}).get('aggregate_sarcasm_label') == 0]

print(f"CSC with label=1: {len(csc_label_1)}")
print(f"CSC with label=0: {len(csc_label_0)}")

# Sample 50 from each category
csc_sample_1 = random.sample(csc_label_1, min(50, len(csc_label_1)))
csc_sample_0 = random.sample(csc_label_0, min(50, len(csc_label_0)))

print(f"Sampled {len(csc_sample_1)} entries with label=1 and {len(csc_sample_0)} entries with label=0 from CSC dataset")
print(f"Total sampled entries from CSC dataset: {len(csc_sample_1) + len(csc_sample_0)}")

# Combine and shuffle
csc_samples = csc_sample_1 + csc_sample_0
random.shuffle(csc_samples)

csc_df_data = []
for entry in csc_samples:
    # Construct dialogue string from turns
    dialogue_text = ""
    turns = entry.get('dialogue', {}).get('turns', [])
    for i, turn in enumerate(turns, 1):
        u1 = turn.get('u1', '')
        u2 = turn.get('u2', '')
        dialogue_text += f"Turn {i}:\n"
        dialogue_text += f"  u1: {u1}\n"

        if i < len(turns):  # Add separator between turns (but not after the last one)
            dialogue_text += f"  u2: {u2}\n"
            dialogue_text += "\n"
    
    csc_df_data.append({
        'unique_id': entry.get('unique_id', ''),
        'dialogue': dialogue_text.strip(),
        'label': entry.get('annotations', {}).get('aggregate_sarcasm_label', ''),
    })

csc_df = pd.DataFrame(csc_df_data)
print(f"\nCSC DataFrame created with {len(csc_df)} rows")



Processing CSC dataset...
CSC with label=1: 973
CSC with label=0: 1973
Sampled 50 entries with label=1 and 50 entries with label=0 from CSC dataset
Total sampled entries from CSC dataset: 100

CSC DataFrame created with 100 rows


In [50]:
# Process MultiPICo dataset
print("\nProcessing MultiPICo dataset...")

# Separate entries by irony_appropriateness_label (inside annotations)
multipico_label_1 = [entry for entry in multipico_data['entries'] if entry.get('annotations', {}).get('irony_appropriateness_label') == 1]
multipico_label_0 = [entry for entry in multipico_data['entries'] if entry.get('annotations', {}).get('irony_appropriateness_label') == 0]

print(f"MultiPICo with label=1: {len(multipico_label_1)}")
print(f"MultiPICo with label=0: {len(multipico_label_0)}")

# Sample 50 from each category
multipico_sample_1 = random.sample(multipico_label_1, min(50, len(multipico_label_1)))
multipico_sample_0 = random.sample(multipico_label_0, min(50, len(multipico_label_0)))

print(f"Sampled {len(multipico_sample_1)} entries with label=1 and {len(multipico_sample_0)} entries with label=0 from MultiPICo dataset")
print(f"Total sampled entries from MultiPICo dataset: {len(multipico_sample_1) + len(multipico_sample_0)}")

# Combine and shuffle
multipico_samples = multipico_sample_1 + multipico_sample_0
random.shuffle(multipico_samples)

multipico_df_data = []
for entry in multipico_samples:
    # Construct dialogue string from turns
    dialogue_text = ""
    turns = entry.get('dialogue', {}).get('turns', [])
    for i, turn in enumerate(turns, 1):
        u1 = turn.get('u1', '')
        u2 = turn.get('u2', '')
        dialogue_text += f"Turn {i}:\n"
        dialogue_text += f"  u1: {u1}\n"
        if i < len(turns):  # Add separator between turns (but not after the last one)
            dialogue_text += f"  u2: {u2}\n"
            dialogue_text += "\n"
    
    multipico_df_data.append({
        'unique_id': entry.get('unique_id', ''),
        'dialogue': dialogue_text.strip(),
        'label': '', #entry.get('annotations', {}).get('irony_appropriateness_label', ''),
    })

multipico_df = pd.DataFrame(multipico_df_data)
print(f"\nMultiPICo DataFrame created with {len(multipico_df)} rows")


Processing MultiPICo dataset...
MultiPICo with label=1: 413
MultiPICo with label=0: 969
Sampled 50 entries with label=1 and 50 entries with label=0 from MultiPICo dataset
Total sampled entries from MultiPICo dataset: 100

MultiPICo DataFrame created with 100 rows


In [51]:
# Create Excel file with two sheets
print("\nCreating Excel file...")

from openpyxl.styles import Alignment

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    csc_df.to_excel(writer, sheet_name='CSC', index=False)
    multipico_df.to_excel(writer, sheet_name='MultiPICo', index=False)
    
    # Format the dialogue column to wrap text
    for sheet_name in ['CSC', 'MultiPICo']:
        worksheet = writer.sheets[sheet_name]
        
        # Find the dialogue column (should be column B)
        dialogue_col = 2  # Column B (1=A, 2=B, 3=C)
        
        # Set column width and wrap text for all cells in dialogue column
        worksheet.column_dimensions['B'].width = 80
        
        for row in range(2, worksheet.max_row + 1):  # Start from row 2 (skip header)
            cell = worksheet.cell(row=row, column=dialogue_col)
            cell.alignment = Alignment(wrap_text=True, vertical='top')

print(f"\nExcel file created successfully at: {output_path}")
print("\nSummary:")
print(f"- CSC sheet: {len(csc_df)} dialogues (50 with label=1, 50 with label=0)")
print(f"- MultiPICo sheet: {len(multipico_df)} dialogues (50 with label=1, 50 with label=0)")



Creating Excel file...

Excel file created successfully at: c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\human\irony_triggering_detection_human_ann.xlsx

Summary:
- CSC sheet: 100 dialogues (50 with label=1, 50 with label=0)
- MultiPICo sheet: 100 dialogues (50 with label=1, 50 with label=0)
